In [1]:
# BPDRR create CONTROLS
# Mario Castro-Gama
# 2026-05-20

In [6]:
import wntr
import numpy as np
import pandas as pd

import random

In [3]:
# this is to remove warnings from showing as pink blocks as results of CELLS
import warnings
warnings.filterwarnings("ignore")

In [30]:
def create_controlled_inp(input_inp, list_pipes, n_crews, output_inp):
    """
    Create a new EPANET INP by adding controls for a list of pipes.

    Parameters
    ----------
    input_inp : str
        Path to original INP file.

    list_pipes : pandas.DataFrame
        DataFrame containing pipe IDs.
        Expected column: 'Pipe ID'

    n_crews : int
        Number of crews available.

    output_inp : str
        Path to output INP file.
    """

    if "Pipe ID" not in list_pipes.columns:
        raise ValueError("list_pipes must contain a column named 'Pipe ID'")

    # Read original INP
    with open(input_inp, "r") as f:
        lines = f.readlines()

    # Locate or create CONTROLS section
    controls_start = None
    next_section = None

    for i, line in enumerate(lines):
        if line.strip().upper() == "[CONTROLS]":
            controls_start = i

            for j in range(i + 1, len(lines)):
                if lines[j].strip().startswith("[") and lines[j].strip().endswith("]"):
                    next_section = j
                    break
            break

    if controls_start is None:
        # Create new CONTROLS section before END
        controls_start = len(lines)
        next_section = len(lines)

        for i, line in enumerate(lines):
            if line.strip().upper() == "[END]":
                controls_start = i
                next_section = i
                break

        lines.insert(controls_start, "\n[CONTROLS]\n")
        controls_start += 1
        next_section += 1

    # Crew availability list
    crews = [0] * n_crews

    new_controls = []
    control_id = 1

    # Assign pipes to crews
    for _, row in list_pipes.iterrows():
        pipe_id = str(row["Pipe ID"])

        # Crew with earliest availability
        crew_idx = min(range(n_crews), key=lambda x: crews[x])

        # Random closure/opening time
        tclose = 0.25*random.randint(0, 673)

        # Store assigned task time in crew slot
        crews[crew_idx] = tclose

        new_controls.append(f"; Crew {crew_idx + 1} - Pipe {pipe_id}\n")
        new_controls.append(f"LINK {pipe_id} OPEN AT TIME {tclose}\n")
        new_controls.append(f"LINK {pipe_id}_A CLOSED AT TIME {tclose}\n")
        new_controls.append(f"LINK {pipe_id}_B CLOSED AT TIME {tclose}\n")
        control_id += 1

    # Insert controls into INP
    #lines = (
    #    lines[:next_section]
    #    + ["\n"] + new_controls + ["\n"]
    #    + lines[next_section:]
    #)
    lines = (
        lines[:next_section]
        + new_controls + ["\n"]
        + lines[next_section:]
    )
    
    # Write output INP
    with open(output_inp, "w") as f:
        f.writelines(lines)

    return crews

In [31]:
# which damage scenario to analyse
ds_sel = 'DS1' 

# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# load the information required 
file_rep = 'BPDRR_reparations.xlsx'

file_distance = 'BBM_distances_'+ds_sel+'.csv'
output_inp = 'BBM-EPS_'+ds_sel+'controls.inp'
FlagSave = 1
FlagPlot = 1

pipes_interest = pd.read_excel(file_rep, 
                               sheet_name=ds_sel,
                               converters={'Junction':str,'Coefficient':float,'Pipe ID':str})
print(pipes_interest.head())
print(pipes_interest.tail())
print(pipes_interest.dtypes)

  Junction          Coefficient Pipe ID
0             E437       15.175     437
1            E3602        2.428    3602
2            E2112        2.428    2112
3            E3562        2.428    3562
4            E5333        2.428    5333
    Junction          Coefficient Pipe ID
120             E614      0.29025     614
121              E69      0.29025      69
122             E740      0.29025     740
123             E880      0.29025     880
124             E940      0.29025     940
Junction             object
Coefficient         float64
Pipe ID              object
dtype: object


In [32]:
crew_schedule = create_controlled_inp(
                                input_inp  = input_inp,
                                list_pipes = pipes_interest,
                                n_crews    = 3,
                                output_inp = output_inp,
                                )

print(crew_schedule)

[9.0, 166.25, 167.75]


In [14]:
pipes_interest.columns

Index(['Junction', 'Coefficient', 'Pipe ID'], dtype='object')

In [27]:
len(np.arange(0,86400+300,300))

289

288.0